# Readmission cohort setup (Supabase)

One-time admin notebook: import cases from parquet, create clinician accounts, assign 30 cases each.

**Prerequisites**
1. Run `supabase/schema.sql` in the Supabase SQL Editor.
2. Copy `notebooks/.env.example` → `notebooks/.env` with your service role key.
3. Place cohort file at `src/data/readmit_30d.parquet`.

```bash
pip install -r notebooks/requirements.txt
```

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from supabase import create_client

load_dotenv(Path("notebooks/.env"))
load_dotenv(Path(".env"))

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

REPO_ROOT = Path.cwd() if (Path.cwd() / "supabase" / "schema.sql").exists() else Path.cwd().parent
PARQUET_PATH = Path(os.getenv("PARQUET_PATH", REPO_ROOT / "src" / "data" / "readmit_30d.parquet"))
CASES_PER_CLINICIAN = int(os.getenv("CASES_PER_CLINICIAN", "30"))
RANDOM_SEED = int(os.getenv("RANDOM_SEED", "42"))

CLINICIANS = [
    {
        "email": "krumholz@yale.edu",
        "password": "welcome",
        "display_name": "Clinician A",
    },
    {
        "email": "karthik@yale.edu",
        "password": "welcome",
        "display_name": "Clinician B",
    },
]

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase client ready")
print("Parquet:", PARQUET_PATH)

Supabase client ready
Parquet: c:\Users\Lough\Desktop\Research\[Yale] Postdoctoral Research\Readmission\src\data\readmit_30d.parquet


In [2]:
def fnv1a_hash(text: str) -> str:
    hash_val = 0x811C9DC5
    for ch in text:
        hash_val ^= ord(ch)
        hash_val = (hash_val * 0x01000193) & 0xFFFFFFFF
    return f"fnv1a-{hash_val:08x}"


def compute_case_version_hash(index_raw: str, readmit_raw: str) -> str:
    index_hash = fnv1a_hash(index_raw)
    readmit_hash = fnv1a_hash(readmit_raw)
    return fnv1a_hash(f"{index_hash}\n---\n{readmit_hash}")


def str_val(v) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return str(v)


def int_val(v) -> int:
    try:
        return int(v)
    except (TypeError, ValueError):
        return 0


def bool_val(v) -> bool | None:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, bool):
        return v
    return str(v).lower() in ("true", "1", "yes")

## Load parquet and select cases

Shuffles the cohort with a fixed seed, takes `CASES_PER_CLINICIAN × len(CLINICIANS)` rows, and splits evenly.

In [3]:
if not PARQUET_PATH.is_file():
    raise FileNotFoundError(f"Cohort file not found: {PARQUET_PATH}")

table = pq.read_table(PARQUET_PATH)
df = table.to_pandas()
df = df[df["row_id"].notna()].copy()
df["row_id"] = df["row_id"].astype(str)

total_needed = CASES_PER_CLINICIAN * len(CLINICIANS)
if len(df) < total_needed:
    raise ValueError(f"Need {total_needed} cases but parquet has {len(df)}")

selected = df.sample(n=total_needed, random_state=RANDOM_SEED).reset_index(drop=True)
splits: list[tuple[dict, list[str]]] = []

for i, clinician in enumerate(CLINICIANS):
    start = i * CASES_PER_CLINICIAN
    end = start + CASES_PER_CLINICIAN
    chunk = selected.iloc[start:end]
    row_ids = chunk["row_id"].tolist()
    splits.append((clinician, row_ids))
    print(f"{clinician['email']}: {len(row_ids)} cases")

print("Sample row_ids:", selected["row_id"].head(3).tolist())

krumholz@yale.edu: 30 cases
karthik@yale.edu: 30 cases
Sample row_ids: ['865', '439', '342']


## Upsert cases

In [4]:
case_rows = []
for _, row in selected.iterrows():
    index_note = str_val(row.get("index_discharge_summary"))
    readmit_note = str_val(row.get("readmit_discharge_summary"))
    case_rows.append(
        {
            "row_id": str_val(row.get("row_id")),
            "patient_identifier": str_val(row.get("patient_identifier")),
            "subject_id": str_val(row.get("subject_id")),
            "index_hadm_id": str_val(row.get("index_hadm_id")),
            "readmit_hadm_id": str_val(row.get("readmit_hadm_id")),
            "index_primary_icd_code": str_val(row.get("index_primary_icd_code")),
            "days_to_readmit": int_val(row.get("days_to_readmit")),
            "readmit_has_icu": bool_val(row.get("readmit_has_icu")),
            "index_discharge_summary": index_note,
            "readmit_discharge_summary": readmit_note,
            "note_version_hash": compute_case_version_hash(index_note, readmit_note),
        }
    )

# Batch upsert in chunks of 20 (large text fields)
BATCH = 20
for i in range(0, len(case_rows), BATCH):
    batch = case_rows[i : i + BATCH]
    sb.table("cases").upsert(batch, on_conflict="row_id").execute()

print(f"Upserted {len(case_rows)} cases")

Upserted 60 cases


## Create auth users + profiles

In [5]:
user_ids: dict[str, str] = {}


def list_all_users():
    users = []
    page = 1
    while True:
        res = sb.auth.admin.list_users(page=page, per_page=200)
        batch = getattr(res, "users", res) or []
        if not batch:
            break
        users.extend(batch)
        if len(batch) < 200:
            break
        page += 1
    return users


existing_users = list_all_users()

for clinician in CLINICIANS:
    email = clinician["email"].lower()
    match = next((u for u in existing_users if getattr(u, "email", "") and u.email.lower() == email), None)

    if match:
        user_id = match.id
        print(f"User exists: {email} ({user_id})")
    else:
        created = sb.auth.admin.create_user(
            {
                "email": email,
                "password": clinician["password"],
                "email_confirm": True,
            }
        )
        user_id = created.user.id
        print(f"Created user: {email} ({user_id})")

    user_ids[email] = user_id
    sb.table("profiles").upsert(
        {
            "id": user_id,
            "display_name": clinician["display_name"],
            "role": "reviewer",
        },
        on_conflict="id",
    ).execute()

print("Profiles upserted for", list(user_ids.keys()))

Created user: krumholz@yale.edu (4f566eef-4836-4084-9e96-ad92863fcf28)
Created user: karthik@yale.edu (e60076f6-571f-42f1-918f-ce1501ffaf4c)
Profiles upserted for ['krumholz@yale.edu', 'karthik@yale.edu']


## Create assignments

In [6]:
assignment_rows = []
manifest = []

for clinician, row_ids in splits:
    email = clinician["email"].lower()
    user_id = user_ids[email]
    for row_id in row_ids:
        assignment_rows.append(
            {
                "user_id": user_id,
                "row_id": row_id,
                "status": "not_started",
            }
        )
        manifest.append({"email": email, "user_id": user_id, "row_id": row_id})

for i in range(0, len(assignment_rows), 50):
    sb.table("assignments").upsert(
        assignment_rows[i : i + 50],
        on_conflict="user_id,row_id",
    ).execute()

print(f"Created {len(assignment_rows)} assignments")

Created 60 assignments


## Verification + manifest export

In [7]:
cases_count = sb.table("cases").select("row_id", count="exact").execute()
assignments_count = sb.table("assignments").select("id", count="exact").execute()

print("Cases in DB:", cases_count.count)
print("Assignments in DB:", assignments_count.count)

for clinician in CLINICIANS:
    email = clinician["email"].lower()
    uid = user_ids[email]
    res = (
        sb.table("assignments")
        .select("row_id", count="exact")
        .eq("user_id", uid)
        .execute()
    )
    print(f"  {email}: {res.count} assignments")

manifest_path = REPO_ROOT / "notebooks" / "assignment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("Wrote manifest:", manifest_path)

Cases in DB: 60
Assignments in DB: 60
  krumholz@yale.edu: 30 assignments
  karthik@yale.edu: 30 assignments
Wrote manifest: c:\Users\Lough\Desktop\Research\[Yale] Postdoctoral Research\Readmission\notebooks\assignment_manifest.json
